# NB11 — S3 manual versus SAM2 mask agreement

**Two-pass notebook. First PREPARE on CPU, then RUN on one Kaggle T4 session.**
Attach Tire Dataset Prepared, enable Internet and HF_TOKEN. Do not run four copies.

This uses **SAM2.1 Hiera Small**, not a changed classifier and not a trained student.
It produces **human-box-prompted, unedited SAM2 masks**, NOT fully automatic semantic
segmentation. Boxes must be drawn from images without consulting manual polygons;
manual-derived prompts or best-reference-IoU mask selection would bias this comparison.
No mask edits or hidden replacement for failed predictions. All 418 clean images
are scored by fold/session; original manual masks are untouched. There are no epochs:
resume is per completed image. Normal HF pushes are every 30 minutes, at major
completion, and on catchable Stop. Hard kernel kills cannot flush unpublished work.

1. Keep MODE=PREPARE; Run All. Download `S3_inputs_to_annotate.zip` from Kaggle Output.
2. In labelme, do the 30-image **blind_pass first**, without viewing old labels.
   Draw tyre/tread polygons and visible marking/damage polygons independently.
3. In boxes/, draw two rectangles (`tyre`, `tread`) per image, not old polygon boxes.
   Keep original filenames/dimensions. These are human prompts, not ground truth masks.
4. Upload only JSON folders boxes/ and blind_pass/ as a **private** Kaggle dataset.
5. Attach it, set INPUT_ROOT to its folder, MODE=RUN and truthfully set both confirmations.
   Select T4 x2; only GPU 0 is used to bound memory. Run All.

The blind tread-IoU gate is the existing >0.90 rule, not a SAM2 acceptance threshold.
SAM2 quality is reported including failures; review it before adopting pseudo-labels
for S5. This notebook does not train S4b/S5/S9 or resolve tyre-fold leakage.


In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow', 'timm'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
    'MCBpZiBzZWxmdGVzdCgpIGVsc2UgMSkK',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


## Session — Internet ON, HF_TOKEN secret; one notebook by default

In [ ]:
# === Who am I? =============================================================
#
# ACCOUNT labels this Kaggle account in the shared run log.
# ACTIVE_KAGGLE_ACCOUNTS is the ONE source of truth for parallelism.
# NUM_WORKERS and WORKER_ID are derived from it; do not edit them.
#
# ---------------------------------------------------------------------------
# THESE TWO VALUES ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide which account owns each fresh run. An absent or partial run stays
# with that static owner. Work stealing is disabled by default; an explicit
# recovery run may opt in and can then take over only a real claim/run event
# older than 45 minutes. Whether a run is finished, and what epoch it reached,
# is read from HuggingFace -- from the run's own files -- so it is the same
# answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
#
# DEFAULT: exactly one Kaggle notebook. For four parallel copies, replace the
# tuple with ('acct1', 'acct2', 'acct3', 'acct4') in every copy and set ACCOUNT
# to that copy's label. Keeping the active labels in one tuple prevents a cell
# that says "one worker" in one place but silently launches as worker 0/4.
ACTIVE_KAGGLE_ACCOUNTS = ('acct1',)   # <<< one notebook; list all four only when all four run
ACCOUNT = 'acct1'                     # <<< this copy's label

# A missing comma in ('acct1') makes it a string. tyrelib deliberately repairs
# that common edit, so a valid one-worker session cannot fail before HF sync.
ACTIVE_KAGGLE_ACCOUNTS = tl.normalise_active_accounts(ACTIVE_KAGGLE_ACCOUNTS)
if ACCOUNT not in ACTIVE_KAGGLE_ACCOUNTS:
    raise ValueError(f"ACCOUNT={ACCOUNT!r} is not active: {ACTIVE_KAGGLE_ACCOUNTS}")
NUM_WORKERS = len(ACTIVE_KAGGLE_ACCOUNTS)
WORKER_ID = ACTIVE_KAGGLE_ACCOUNTS.index(ACCOUNT)
print(f"RUN MODE CHECK: {ACCOUNT=} {ACTIVE_KAGGLE_ACCOUNTS=} -> worker {WORKER_ID}/{NUM_WORKERS}")

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='s3',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


In [ ]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


In [ ]:
# First run PREPARE on CPU. RUN needs your independent inputs and one T4.
MODE = "PREPARE"  # change to "RUN" after completing the image-only input package
INPUT_ROOT = Path("/kaggle/input/s3-independent-inputs")
# Required only in RUN: affirm the actual procedure, not merely to bypass a check.
BOXES_DRAWN_FROM_IMAGES_ONLY = False
BLIND_PASS_WITHOUT_VIEWING_OLD_MASKS = False
# No four-worker inference: one session owns this mask arm.
assert NUM_WORKERS == 1

S3_IMPLEMENTATION_SHA = "7fd8d0c5b4e40ae940d2d7481aa84193d127c7f910d831be5f74dda267a26a4d"


In [ ]:
"""Pure S3 input/metric/resume helpers, embedded in NB11. No model or HF imports."""
import hashlib
import json
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw

SAM_COMMIT = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
MODEL_REVISION = 'ee5bba1d82bb8749febdf90f45e84b687142ba03'
MODEL_ID = 'facebook/sam2.1-hiera-small'

def digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def signature(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def selfcheck_ids(frame):
    # Fixed before seeing any mask or score; ten images in each original fold.
    ids = []
    for fold, group in frame.groupby('fold_id', sort=True):
        ids += sorted(group.image_id, key=lambda x: signature(['s3-blind-4622', x]))[:10]
    assert len(ids) == 30 and len(set(ids)) == 30
    return ids

def regions(mask):
    if mask.ndim != 2 or not np.isin(mask, [0,1,2,3,4]).all():
        raise ValueError('Expected a native-sized indexed mask with labels 0..4')
    return {'tyre': mask > 0, 'tread': (mask == 2) | (mask == 3)}

def scores(a, b):
    if a.shape != b.shape:
        raise ValueError('Mask dimensions differ; do not silently resize annotations')
    a, b = a.astype(bool), b.astype(bool)
    intersection, union = (a & b).sum(), (a | b).sum()
    total = a.sum() + b.sum()
    return dict(iou=float(intersection/union) if union else None,
                dice=float(2*intersection/total) if total else None,
                reference_pixels=int(a.sum()), prediction_pixels=int(b.sum()))

def read_boxes(path, width, height):
    d = json.loads(Path(path).read_text())
    if (d['imageWidth'], d['imageHeight']) != (width, height):
        raise ValueError(f'Wrong original image dimensions: {path}')
    out = {}
    for shape in d.get('shapes', []):
        name = shape.get('label', '').strip().lower()
        if name not in ('tyre', 'tread') or name in out or shape.get('shape_type') != 'rectangle':
            raise ValueError('Prompts require exactly two rectangles labelled tyre and tread; no polygons/duplicates')
        xy = np.asarray(shape['points'], dtype=float)
        if xy.shape != (2,2) or not np.isfinite(xy).all():
            raise ValueError('Invalid rectangle')
        lo, hi = xy.min(0), xy.max(0)
        if not (0 <= lo[0] < hi[0] <= width and 0 <= lo[1] < hi[1] <= height):
            raise ValueError('Rectangle outside image or empty')
        out[name] = [*lo.tolist(), *hi.tolist()]
    if set(out) != {'tyre','tread'}:
        raise ValueError('Both tyre and tread rectangles are required')
    return out

def read_blind(path, width, height):
    d = json.loads(Path(path).read_text())
    if (d['imageWidth'], d['imageHeight']) != (width,height):
        raise ValueError('Blind pass dimensions mismatch')
    labels = {'tyre':1, 'tread':2, 'marking':3, 'damage':4}
    shapes = d.get('shapes', [])
    if not {'tyre','tread'} <= {s.get('label','').strip().lower() for s in shapes}:
        raise ValueError('Blind pass must include tyre and tread polygons')
    im = Image.new('L',(width,height)); draw = ImageDraw.Draw(im)
    for s in shapes:
        if s.get('label','').strip().lower() not in labels or s.get('shape_type','polygon') != 'polygon':
            raise ValueError('Blind pass accepts only canonical label polygons')
    for label, value in labels.items():
        for s in shapes:
            if s['label'].strip().lower() == label:
                p = np.asarray(s['points'],dtype=float)
                if p.ndim != 2 or p.shape[1] != 2 or len(p) < 3 or not np.isfinite(p).all():
                    raise ValueError('Invalid blind polygon')
                if (p < 0).any() or (p[:,0] > width).any() or (p[:,1] > height).any():
                    raise ValueError('Blind polygon outside image')
                draw.polygon([tuple(x) for x in p], fill=value)
    return np.asarray(im)

def load_record(path, key, shape):
    with np.load(path, allow_pickle=False) as z:
        meta = json.loads(str(z['metadata'].item()))
        if meta['key'] != key:
            raise ValueError('Saved record protocol/input fingerprint mismatch')
        masks = {k: z[k].copy() for k in ('tyre','tread')}
        if any(m.shape != shape or m.dtype != np.bool_ for m in masks.values()):
            raise ValueError('Saved record mask invalid')
    return masks, meta

def save_record(path, masks, meta):
    path = Path(path)
    tmp = path.with_suffix('.partial')
    with tmp.open('wb') as f:
        np.savez_compressed(f, **masks, metadata=json.dumps(meta, sort_keys=True))
    tmp.replace(path)

def consistency_gate(rows):
    tread = [r['iou'] for r in rows if r['region'] == 'tread']
    if len(tread) != 30 or any(v is None or not np.isfinite(v) for v in tread):
        return 'pending'
    return 'pass' if float(np.mean(tread)) > .90 else 'fail'


In [ ]:
import json, hashlib, zipfile, shutil
import numpy as np, pandas as pd
from PIL import Image
clean = pd.read_csv(Path(DATA_ROOT)/"manifests/clean_manifest.csv").sort_values("image_id")
assert len(clean)==418 and clean.image_id.is_unique
assert ANN_ROOT is not None, "Attach the existing annotations beside FINAL"
manual_root = Path(ANN_ROOT)/"clean/masks"
assert manual_root.is_dir()
blind_ids = selfcheck_ids(clean)
if MODE == "PREPARE":
    # Download this PRIVATE local output; never enqueue source images to public HF.
    output = Path('/kaggle/working/S3_inputs_to_annotate.zip')
    total = sum((Path(DATA_ROOT)/r.relative_path).stat().st_size for r in clean.itertuples())
    assert total < 2*1024**3 and shutil.disk_usage(output.parent).free > total*2 + 1024**3
    with zipfile.ZipFile(output, 'w', compression=zipfile.ZIP_STORED) as z:
        for r in clean.itertuples():
            source = Path(DATA_ROOT)/r.relative_path
            z.write(source, f'boxes/{r.image_id}{source.suffix}')
            if r.image_id in blind_ids:
                z.write(source, f'blind_pass/{r.image_id}{source.suffix}')
        z.writestr('INSTRUCTIONS.txt', 'Use labelme locally. boxes/: draw exactly two RECTANGLES tyre and tread on each image, no old masks. blind_pass/: independently draw tyre/tread polygons, marking/damage when visible, WITHOUT opening old masks. Disable embedded imageData on export. Upload JSON only as a PRIVATE Kaggle dataset with boxes/ and blind_pass/ folders. Do blind_pass FIRST, before viewing any old annotations or model results.')
    print('PREPARE COMPLETE. Download', output, 'and finish human inputs; no GPU work started.')
elif MODE != "RUN":
    raise ValueError('MODE must be PREPARE or RUN')


In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
import re, time
from email.utils import parsedate_to_datetime
from huggingface_hub import HfApi, hf_hub_download
REPO = tl.HF_REPO_DEFAULT
READ_TOKEN = getattr(sess.uploader, "token", None)
if not READ_TOKEN:
    raise RuntimeError("HF_TOKEN is required: enable the Kaggle secret and rerun the Session cell.")

def hf_read(label, operation, attempts=8):
    # API metadata calls need retries too; uploads retain their existing schedule.
    for attempt in range(attempts):
        try:
            return operation()
        except Exception as exc:
            response = getattr(exc, "response", None)
            status = getattr(response, "status_code", None)
            if status not in (429, 500, 502, 503, 504):
                raise
            if attempt == attempts - 1:
                raise RuntimeError(f"HF read {label} still unavailable after {attempts} attempts; rerun this cell later.") from None
            headers = {k.lower(): v for k, v in getattr(response, "headers", {}).items()}
            delays = [min(300, 5 * 2**attempt)]
            hint = headers.get("retry-after", "")
            if hint:
                try:
                    delays.append(float(hint))
                except ValueError:
                    try:
                        delays.append(parsedate_to_datetime(hint).timestamp() - time.time())
                    except (ValueError, TypeError, OverflowError):
                        pass
            for match in re.finditer(r'\bt\s*=\s*(\d+)', headers.get("ratelimit", "")):
                delays.append(float(match.group(1)))
            body_delay = tl.parse_retry_after(str(exc))
            if body_delay is not None:
                delays.append(body_delay)
            remaining = max(delays) + 2
            print(f"[HF read] {label}: HTTP {status}; waiting {remaining:.0f}s, retry {attempt+1}/{attempts-1}.", flush=True)
            # Short sleeps allow Kaggle Stop to interrupt the wait normally.
            while remaining > 0:
                step = min(10, remaining)
                time.sleep(step)
                remaining -= step



In [ ]:
if MODE == "RUN":
    import os, sys, subprocess, time, gc, platform
    import torch
    from packaging.version import Version
    from huggingface_hub import HfApi, hf_hub_download
    assert BOXES_DRAWN_FROM_IMAGES_ONLY and BLIND_PASS_WITHOUT_VIEWING_OLD_MASKS, "Complete and truthfully confirm independent inputs first"
    assert torch.cuda.is_available(), 'Select GPU T4 x2 (one GPU used, no training)'
    assert Version(torch.__version__.split('+')[0]) >= Version('2.5.1'), 'Use a current Kaggle image with torch >=2.5.1; do not hot-replace torch'
    # Validate every input BEFORE downloading a model or starting inference.
    boxes, reference_hashes, image_hashes, blind = {}, {}, {}, {}
    for r in clean.itertuples():
        image = Path(DATA_ROOT)/r.relative_path
        with Image.open(image) as im:
            w,h = im.size
        boxes[r.image_id] = read_boxes(INPUT_ROOT/'boxes'/f'{r.image_id}.json', w,h)
        image_hashes[r.image_id] = digest(image)
        mp = manual_root/f'{r.image_id}.png'
        reference_hashes[r.image_id] = digest(mp)
        with Image.open(mp) as im:
            assert np.asarray(im).shape == (h,w)
            regions(np.asarray(im))
        if r.image_id in blind_ids:
            bp = INPUT_ROOT/'blind_pass'/f'{r.image_id}.json'
            blind[r.image_id] = digest(bp)
            read_blind(bp,w,h)
    protocol = dict(revision='s3-box-sam21-small-r1', code=SAM_COMMIT, model=MODEL_ID,
        model_revision=MODEL_REVISION, prompts=boxes, images=image_hashes,
        references=reference_hashes, blind_hashes=blind, blind_ids=blind_ids,
        mask_selection='maximum SAM predicted IoU; never reference IoU', precision='fp32',
        human_box_prompted=True, manual_mask_correction=False, seed=4622,
        self_consistency_gate='mean tread IoU > 0.90 on all 30 fixed images',
        torch=torch.__version__, numpy=np.__version__, implementation=S3_IMPLEMENTATION_SHA)
    arm = signature(protocol)
    prefix = f's3/{protocol["revision"]}/{arm}'
    root = Path(sess.stage_dir)/'s3'/arm
    (root/'records').mkdir(parents=True,exist_ok=True)
    def publish(path):
        sess.uploader.enqueue(path, prefix+'/'+Path(path).relative_to(root).as_posix())
    (root/'protocol.json').write_text(json.dumps(protocol,indent=2))
    publish(root/'protocol.json')
    assert sess.push_now('S3 protocol frozen before inference'), 'HF protocol upload failed; rerun later'
    revision = hf_read('S3 resume revision', lambda: HfApi().repo_info(REPO,repo_type='dataset',token=READ_TOKEN)).sha
    files = set(hf_read('S3 resume inventory', lambda: HfApi().list_repo_files(REPO,repo_type='dataset',revision=revision,token=READ_TOKEN)))
    # Restore only this exact fingerprint. Different prompts/inputs cannot reuse old masks.
    pending = []
    for r in clean.itertuples():
        path = root/'records'/f'{r.image_id}.npz'
        remote = prefix+'/records/'+path.name
        if not path.exists() and remote in files:
            downloaded = hf_read('restore '+r.image_id, lambda: hf_hub_download(REPO,remote,repo_type='dataset',revision=revision,token=READ_TOKEN,local_dir=str(root/'restore')))
            shutil.copyfile(downloaded,path)
        if path.exists():
            load_record(path, signature([arm,r.image_id]), (r.height,r.width))
            if remote not in files:
                publish(path)  # retry unpublished local records after an interrupted flush
        else:
            pending.append(r)
    print('HF/local valid records:', 418-len(pending), 'remaining:',len(pending))
    started = time.monotonic()
    if pending:
        # Pin upstream source. Do not upgrade Kaggle torch/torchvision or compile CUDA extensions.
        os.environ['SAM2_BUILD_CUDA'] = '0'
        subprocess.run([sys.executable,'-m','pip','install','-q','hydra-core==1.3.2','iopath==0.1.10'],check=True)
        subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','--no-build-isolation',
            f'git+https://github.com/facebookresearch/sam2.git@{SAM_COMMIT}'],check=True)
        from sam2.build_sam import build_sam2
        from sam2.sam2_image_predictor import SAM2ImagePredictor
        ck = hf_read('pinned SAM2 checkpoint', lambda: hf_hub_download(MODEL_ID,'sam2.1_hiera_small.pt',revision=MODEL_REVISION,token=READ_TOKEN,local_dir=str(root/'model_cache')))
        torch.manual_seed(4622); np.random.seed(4622)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        predictor = SAM2ImagePredictor(build_sam2('configs/sam2.1/sam2.1_hiera_s.yaml',ck,device='cuda:0',apply_postprocessing=False))
    failure = None
    try:
        for index,r in enumerate(pending):
            if time.monotonic()-started > 7.5*3600:
                raise RuntimeError('Planned session pause; rerun RUN to restore completed image records')
            assert shutil.disk_usage(root).free > 2*1024**3, 'Low staging disk: pause safely'
            with Image.open(Path(DATA_ROOT)/r.relative_path) as im:
                rgb = np.asarray(im.convert('RGB'))
            masks, confidence = {}, {}
            with torch.inference_mode():
                predictor.set_image(rgb)
                for name in ('tyre','tread'):
                    m, q, _ = predictor.predict(box=np.array(boxes[r.image_id][name],dtype=np.float32),multimask_output=True)
                    assert np.isfinite(q).all()
                    best = int(np.argmax(q))
                    masks[name] = m[best].astype(bool)
                    confidence[name] = float(q[best])
            predictor.reset_predictor()
            meta = dict(key=signature([arm,r.image_id]),image_id=r.image_id,arm=arm,confidence=confidence,
                fold=int(r.fold_id),session=r.session_group)
            path = root/'records'/f'{r.image_id}.npz'
            save_record(path,masks,meta); publish(path)
            del rgb, masks, m
            gc.collect(); torch.cuda.empty_cache()
            print(f'{index+1}/{len(pending)} new masks saved: {r.image_id}',flush=True)
            sess.maybe_push('S3 30-minute checkpoint',min_gap_min=30)
    except BaseException as exc:
        failure = dict(type=type(exc).__name__, completed=len(list((root/'records').glob('*.npz'))))
        (root/'last_interruption.json').write_text(json.dumps(failure)); publish(root/'last_interruption.json')
        raise
    finally:
        sess.push_now('S3 inference complete or interrupted')
        if 'predictor' in globals():
            del predictor
        gc.collect(); torch.cuda.empty_cache()
    # Manual references are read ONLY for evaluation, not prediction or mask selection.
    rows, self_rows = [], []
    for r in clean.itertuples():
        pred, meta = load_record(root/'records'/f'{r.image_id}.npz',signature([arm,r.image_id]),(r.height,r.width))
        with Image.open(manual_root/f'{r.image_id}.png') as im:
            refs = regions(np.asarray(im))
        for name in ('tyre','tread'):
            rows.append(dict(image_id=r.image_id,fold=r.fold_id,session=r.session_group,region=name,
                **scores(refs[name],pred[name]),confidence=meta['confidence'][name],
                tread_outside_tyre=int((pred['tread'] & ~pred['tyre']).sum())))
        if r.image_id in blind_ids:
            second_path = INPUT_ROOT/'blind_pass'/f'{r.image_id}.json'
            second = regions(read_blind(second_path,r.width,r.height))
            # Save indexed masks, never embedded RGB from labelme JSON.
            dest = root/f'blind_{r.image_id}.png'
            Image.fromarray(read_blind(second_path,r.width,r.height)).save(dest); publish(dest)
            for name in ('tyre','tread'):
                self_rows.append(dict(image_id=r.image_id,region=name,**scores(refs[name],second[name])))
    metrics, consistency = pd.DataFrame(rows),pd.DataFrame(self_rows)
    metrics.to_csv(root/'agreement_by_image.csv',index=False)
    consistency.to_csv(root/'self_consistency_by_image.csv',index=False)
    for grouping,name in [(['region'],'overall'),(['fold','region'],'by_fold'),(['session','region'],'by_session')]:
        metrics.groupby(grouping).agg(n=('image_id','size'),n_defined=('iou','count'),mean_iou=('iou','mean'),min_iou=('iou','min'),mean_dice=('dice','mean')).to_csv(root/f'agreement_{name}.csv')
    status = dict(mask_generation='complete',images=418,comparison_rows=len(metrics),
        self_consistency=consistency_gate(self_rows),selfcheck_images=30,
        pseudo_label_quality='reported_not_automatically_approved_for_S5',
        human_box_prompted=True,not_fully_automatic=True,arm=arm,
        s3_status='review_required',s5_started=False)
    (root/'STATUS.json').write_text(json.dumps(status,indent=2))
    for path in list(root.glob('*.csv'))+[root/'STATUS.json']:
        publish(path)
    assert sess.finish(), 'Upload incomplete; rerun to retry publication, not regenerate valid records'
    latest = hf_read('final verification',lambda:HfApi().repo_info(REPO,repo_type='dataset',token=READ_TOKEN)).sha
    inventory = set(hf_read('verify artifacts',lambda:HfApi().list_repo_files(REPO,repo_type='dataset',revision=latest,token=READ_TOKEN)))
    required = [prefix+'/records/'+r.image_id+'.npz' for r in clean.itertuples()]
    required += [prefix+'/'+p.name for p in root.glob('*.csv')]+[prefix+'/STATUS.json',prefix+'/protocol.json']
    assert set(required)<=inventory, 'Some expected results missing on HF; rerun RUN to republish'
    print('VERIFIED HF:',latest,prefix)
    print(metrics.groupby('region')[['iou','dice']].mean())
    print(status)
else:
    print('Preparation only. S3 is NOT completed until independent inputs, GPU inference and review.')
